In [1]:
# Import libraries
import tensorflow as tf
import pandas as pd
import time

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense

In [2]:
# 1. Load IMDB Dataset
vocab_size = 10000
max_length = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=vocab_size
)

print("Training samples:", len(x_train))
print("Testing samples:", len(x_test))

Training samples: 25000
Testing samples: 25000


In [3]:
# 2. Preprocess Dataset
x_train = pad_sequences(
    x_train, maxlen=max_length,
    padding="post", truncating="post"
)

x_test = pad_sequences(
    x_test, maxlen=max_length,
    padding="post", truncating="post"
)

In [4]:
# 3. Build Model
def build_model(architecture):
    model = Sequential()
    model.add(Embedding(vocab_size, 128, mask_zero=True))

    if architecture == "RNN":
        model.add(SimpleRNN(64))
    elif architecture == "LSTM":
        model.add(LSTM(64))
    else:
        model.add(GRU(64))

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [ ]:
# 4. Train and Evaluate Models
architectures = ["RNN", "LSTM", "GRU"]
results = []

for architecture in architectures:

    print("\nTraining model:", architecture)

    model = build_model(architecture)

    start_time = time.time()

    model.fit(
        x_train,
        y_train,
        epochs=6,
        batch_size=64,
        validation_split=0.2,
        verbose=1
    )

    training_time = time.time() - start_time

    test_loss, test_accuracy = model.evaluate(
        x_test, y_test, verbose=0
    )

    results.append({
        "Model": architecture,
        "Test Accuracy (%)": round(test_accuracy * 100, 2),
        "Test Loss": round(test_loss, 4),
        "Training Time (seconds)": round(training_time, 2)
    })

Training model: RNN
Epoch 1/6
313/313 ━━━━━━━━━━━━━━━━━━━━ 15s 44ms/step - accuracy: 0.6230 - loss: 0.6347 - val_accuracy: 0.6560 - val_loss: 0.6107
Epoch 2/6
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.6748 - loss: 0.5907 - val_accuracy: 0.6358 - val_loss: 0.6271
Epoch 3/6
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.7571 - loss: 0.5002 - val_accuracy: 0.6172 - val_loss: 0.6473
Epoch 4/6
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.7868 - loss: 0.4541 - val_accuracy: 0.6578 - val_loss: 0.6309
Epoch 5/6
313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 44ms/step - accuracy: 0.8159 - loss: 0.3982 - val_accuracy: 0.6258 - val_loss: 0.7111
Epoch 6/6
313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 45ms/step - accuracy: 0.8573 - loss: 0.3334 - val_accuracy: 0.6004 - val_loss: 0.7569

Training model: LSTM
Epoch 1/6
313/313 ━━━━━━━━━━━━━━━━━━━━ 32s 98ms/step - accuracy: 0.7509 - loss: 0.5097 - val_accuracy: 0.8184 - val_loss: 0.4163
Epoch 2/6
313/313 ━━━━━━━━━━━━━━━━━━━━ 31s 100ms/step -

In [ ]:
# 5. Display Comparison Table
results_df = pd.DataFrame(results)

print("\n" + "=" * 75)
print("          RNN, LSTM AND GRU PERFORMANCE COMPARISON")
print("=" * 75)

print(
    f"{'Model':<15}"
    f"{'Test Accuracy (%)':>20}"
    f"{'Test Loss':>15}"
    f"{'Training Time (s)':>23}"
)

print("-" * 75)

for _, row in results_df.iterrows():
    print(
        f"{row['Model']:<15}"
        f"{row['Test Accuracy (%)']:>20.2f}"
        f"{row['Test Loss']:>15.4f}"
        f"{row['Training Time (seconds)']:>23.2f}"
    )

print("=" * 75)

          RNN, LSTM AND GRU PERFORMANCE COMPARISON
Model             Test Accuracy (%)      Test Loss      Training Time (s)
---------------------------------------------------------------------------
RNN                           59.39         0.7679                  83.08
LSTM                          83.20         0.5411                 187.20
GRU                           83.09         0.5707                 172.75


In [ ]:
# 6. Final Analysis
best_accuracy_model = results_df.loc[
    results_df["Test Accuracy (%)"].idxmax(), "Model"
]

lowest_loss_model = results_df.loc[
    results_df["Test Loss"].idxmin(), "Model"
]

print("\n" + "=" * 35)
print("          FINAL ANALYSIS")
print("=" * 35)
print(f"  Best Test Accuracy : {best_accuracy_model}")
print(f"  Lowest Test Loss   : {lowest_loss_model}")
print("=" * 35)

          FINAL ANALYSIS
  Best Test Accuracy : LSTM
  Lowest Test Loss   : LSTM
